<a href="https://colab.research.google.com/github/mohamadfaisalbashir/grokking-Deep-Learning/blob/main/10_neural_learning_about_edges_and_corners.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Chapter 10: Neural Learning about Edges and Corners - Intro to Convolutional Neural Networks**

This notebook summarizes Chapter 10 of *Grokking Deep Learning*:

1. Reusing Weights in Multiple Places
2. The Convolutional Layer
3. A Simple Implementation in NumPy
4. Summary

This chapter introduces the **convolutional layer**, the best-known example of a general principle: when a network has to detect the same pattern in several places, it should use the **same weights** in all of those places. The chapter first explains why such weight reuse fights overfitting, then describes how a convolutional layer is built from many very small linear layers (kernels) that are applied at every position of an image, and finally shows a NumPy implementation in which the first layer of the MNIST network of the previous chapter is replaced by a convolutional layer.

The chapter opens with a remark by Geoffrey Hinton (from an "Ask Me Anything" session on Reddit), who calls the pooling operation of convolutional networks a big mistake and finds it troubling that it nevertheless works so well.

---
## **1. Reusing Weights in Multiple Places**

**If you need to detect the same feature in multiple places, use the same weights!**

### **1.1 Overfitting Revisited**

The greatest challenge in neural networks is **overfitting**: the network memorizes the training set instead of learning useful abstractions that generalize to unseen data. In other words, it bases its predictions on the noise in the dataset rather than on the fundamental signal (the book recalls the earlier analogy of a fork pressed into clay).

Overfitting is often caused by having **more parameters than necessary** for a given dataset. A network with that many parameters can memorize every fine-grained detail of the training set. The book illustrates this with two imaginary network "thoughts" about a digit. A memorizing network says, in effect, "I have seen image number 363 again; it was a 2". A network that has learned abstractions says instead, "there is a swooping top, a swirl at the bottom left and a tail on the right, so this must be a 2". When a network has many parameters but there are not many training examples, overfitting is hard to avoid.

Chapter 8 dealt with this problem at length through **regularization**. Regularization, however, is neither the only technique against overfitting nor even the most desirable one.

### **1.2 Structure: Reducing the Weight-to-Data Ratio**

Overfitting depends on the **ratio between the number of weights in the model and the number of datapoints** available to learn those weights. A better remedy is therefore, whenever possible, to use something the book loosely calls **structure**.

**Structure** means deliberately reusing the same weights for several purposes inside a network, because the same pattern is believed to be needed in several places. Reusing weights lowers the weight-to-data ratio, which can considerably reduce overfitting and lead to far more accurate models.

> Two regions of a handwritten 3 (circled) contain similar strokes: the same kind of pattern can occur in more than one place of an image.
>
> <img src="pictures/chapter 10/picture 10.1.png" alt="Figure 10.1" width="600"/>

Normally, removing parameters makes a model less expressive, that is, less able to learn patterns. If the reuse of weights is placed cleverly, however, the model remains **equally expressive while becoming more robust against overfitting**. As a pleasant side effect, the model also tends to become smaller, because fewer distinct parameters have to be stored.

The most famous and most widely used structure in neural networks is the **convolution**; when it is used as a layer, it is called a **convolutional layer**.

---
## **2. The Convolutional Layer**

**Lots of very small linear layers are reused in every position, instead of a single big one.**

### **2.1 From One Big Layer to Many Small Ones**

A conventional dense linear layer connects every input to every output. The central idea of a convolutional layer is different: instead of one large dense layer, it uses **many very small linear layers**, usually with **fewer than 25 inputs and a single output**, and applies each of them at **every input position**. Each of these mini-layers is called a **convolutional kernel**. A kernel is nothing more than a "baby" linear layer with a small number of inputs and one output.

### **2.2 A Kernel Scanning an Image**

> A single 3 x 3 kernel placed at the top-left corner of an 8 x 8 image: its nine inputs see the pixel values 0 0 0 / 0 0 0 / 0 0 1 and it produces a single output.
>
> <img src="pictures/chapter 10/picture 10.2.png" alt="Figure 10.2" width="600"/>

The figure shows one 3 x 3 convolutional kernel. The kernel makes a prediction at its current location, moves one pixel to the right, predicts again, moves another pixel, and so on. After it has reached the right edge of the image, it moves down by a single pixel and scans back towards the left, repeating this until it has made a prediction at **every possible position** in the image.

The result is a **smaller square of kernel predictions**, which serves as the input of the next layer. A convolutional layer normally contains **many kernels**, each producing its own square of predictions.

### **2.3 Several Kernels and Pooling**

> Bottom to top: four 3 x 3 kernels predict over the same 8 x 8 image of a 2; each produces a 6 x 6 matrix of outputs; the position-wise maximum of the four matrices forms the final 6 x 6 matrix that is passed on to the next layer.
>
> <img src="pictures/chapter 10/picture 10.3.png" alt="Figure 10.3" width="600"/>

The figure shows four different convolutional kernels processing the same 8 x 8 image of a 2. Each kernel yields a 6 x 6 prediction matrix, so the convolutional layer with four 3 x 3 kernels produces **four 6 x 6 prediction matrices**. These matrices can be combined element by element in three ways:

| Combination | Operation on the four matrices |
|---|---|
| **Sum pooling** | add the matrices elementwise |
| **Mean pooling** | take the elementwise mean |
| **Max pooling** | take the elementwise maximum |

**Max pooling** is the most popular variant. For each position, one looks at the outputs of the four kernels, finds the largest value and copies it into a final 6 x 6 matrix (shown at the top of the figure). **Only this final matrix** is forward propagated into the following layers. (This pooling operation is the one that Hinton's remark at the beginning of the chapter refers to.)

### **2.4 What the Kernels Detect**

The book points out a few things in the figure:

- One kernel forward propagates a 1 only when it is focused on a **horizontal line segment**.
- Another kernel forward propagates a 1 only when it is focused on a **diagonal line pointing upward and to the right**.
- One kernel did **not find any of the patterns** it was trained to predict, so its output matrix is empty. (The remaining kernel fires at a single position of this image.)

### **2.5 Why This Reduces Overfitting**

Each kernel can learn **one particular pattern** and then **search for its presence anywhere in the image**. A single small set of weights is therefore trained on a much larger number of examples: although the dataset itself has not changed, each mini-kernel is forward propagated many times, on many segments of the data. This changes the **ratio between the number of weights and the number of datapoints** on which those weights are trained. The effect on the network is powerful: its ability to overfit the training data drops drastically, while its ability to generalize increases.

---
## **3. A Simple Implementation in NumPy**

**Just think mini-linear layers, and you already know what you need to know.**

The implementation starts with forward propagation. The book builds it in small pieces (three fragments) and then prints the **entire program** as one listing.

### **3.1 Selecting a Subregion of a Batch of Images**

The following function selects a subregion from a batch of images. Note that it selects the **same subregion for every image of the batch**.

```python
def get_image_section(layer,row_from, row_to, col_from, col_to):
    sub_section = layer[:,row_from:row_to,col_from:col_to]
    return subsection.reshape(-1,1,row_to-row_from, col_to-col_from)
```

*Note:* the book's first fragment (shown above exactly as printed) assigns the slice to `sub_section` but returns `subsection`, so calling it would raise a `NameError`. The complete listing in Section 3.4 prints the function with the consistent variable name `section`, and that is the version used in the code cell of this notebook.

### **3.2 Visiting Every Location of the Image**

Because the function extracts only one subregion of the batch, it has to be called many times, once for every location in the image. The following loop does this:

```python
layer_0 = images[batch_start:batch_end]
layer_0 = layer_0.reshape(layer_0.shape[0],28,28)
layer_0.shape

sects = list()
for row_start in range(layer_0.shape[1]-kernel_rows):
    for col_start in range(layer_0.shape[2] - kernel_cols):
        sect = get_image_section(layer_0,
                                 row_start,
                                 row_start+kernel_rows,
                                 col_start,
                                 col_start+kernel_cols)
        sects.append(sect)

expanded_input = np.concatenate(sects,axis=1)
es = expanded_input.shape
flattened_input = expanded_input.reshape(es[0]*es[1],-1)
```

Here `layer_0` is a batch of images of shape 28 x 28. The `for` loops visit every `kernel_rows` x `kernel_cols` subregion of the images and collect them in the list `sects`. This list of sections is then concatenated and reshaped in a slightly unusual way. (These fragments are not run as separate cells, because they use variables such as `images`, `batch_start` and `kernel_rows` that are defined only in the complete listing below. All of their lines reappear there.)

### **3.3 Treating Every Subregion as Its Own Image**

The trick is to **pretend that each individual subregion is a separate image**. If a batch contains 8 images and each image has 100 subregions, one pretends to have a batch of 800 smaller images. Forward propagating them through a linear layer with **one output neuron** is exactly the same as applying that linear layer at every subregion of every image in the batch.

If the linear layer has **n output neurons** instead, it produces the same outputs as n linear layers (n kernels) applied at every position of the image. The book implements it this way because the code becomes both **simpler and faster**:

```python
kernels = np.random.random((kernel_rows*kernel_cols,num_kernels))
               ...
kernel_output = flattened_input.dot(kernels)
```

(The line of dots is part of the book's fragment; it stands for the code in between.) In other words, the weights of all kernels form one matrix with `kernel_rows*kernel_cols` rows and `num_kernels` columns, and one matrix product computes every kernel at every position.

### **3.4 The Complete Listing**

The book now prints the entire NumPy implementation. The following points should be noted before running it:

- According to the book, it replaces the first layer of the network from Chapter 9 by a convolutional layer. Its other pieces (`tanh`, `tanh2deriv`, `softmax`, the one-hot labels, batches, a random mask on the hidden layer) are part of the listing itself, so the cells are self-contained.
- The listing **needs the `keras` package**: MNIST is loaded with `from keras.datasets import mnist` and `mnist.load_data()`. The network is trained on the **first 1,000 training images**, and after every iteration it is evaluated on the **entire test set**. The notebook does not contain any download code.
- Because the test images are processed one at a time in a Python loop in every one of the 300 iterations, a full run takes a long time.

**Settings and shapes.**

| Name | Value | Meaning |
|---|---|---|
| `alpha`, `iterations` | 2, 300 | learning rate and number of passes |
| `batch_size` | 128 | images per batch |
| `input_rows`, `input_cols` | 28, 28 | size of an MNIST image |
| `kernel_rows`, `kernel_cols` | 3, 3 | size of each kernel |
| `num_kernels` | 16 | number of kernels in the convolutional layer |
| `num_labels` | 10 | number of digit classes |
| `hidden_size` | ((28 - 3) * (28 - 3)) * 16 = 10,000 | number of outputs of the convolutional layer, which is the input size of `weights_1_2` |

The kernels are a small matrix `kernels` of shape (9, 16) with random starting values between -0.01 and 0.01. The matrix `weights_1_2` of shape (10000, 10) has random starting values between -0.1 and 0.1 and maps the convolutional output to the ten labels. The first cell below contains the imports, the data preparation, the helper functions, the settings and the function `get_image_section`.

In [1]:
import numpy as np, sys
np.random.seed(1)

from keras.datasets import mnist

(x_train, y_train), (x_test, y_test) = mnist.load_data()

images, labels = (x_train[0:1000].reshape(1000,28*28) / 255,
                  y_train[0:1000])


one_hot_labels = np.zeros((len(labels),10))
for i,l in enumerate(labels):
    one_hot_labels[i][l] = 1
labels = one_hot_labels

test_images = x_test.reshape(len(x_test),28*28) / 255
test_labels = np.zeros((len(y_test),10))
for i,l in enumerate(y_test):
    test_labels[i][l] = 1

def tanh(x):
    return np.tanh(x)

def tanh2deriv(output):
    return 1 - (output ** 2)

def softmax(x):
    temp = np.exp(x)
    return temp / np.sum(temp, axis=1, keepdims=True)

alpha, iterations = (2, 300)
pixels_per_image, num_labels = (784, 10)
batch_size = 128

input_rows = 28
input_cols = 28

kernel_rows = 3
kernel_cols = 3
num_kernels = 16

hidden_size = ((input_rows - kernel_rows) *
               (input_cols - kernel_cols)) * num_kernels

kernels = 0.02*np.random.random((kernel_rows*kernel_cols,
                                 num_kernels))-0.01

weights_1_2 = 0.2*np.random.random((hidden_size,
                                    num_labels)) - 0.1

def get_image_section(layer,row_from, row_to, col_from, col_to):
    section = layer[:,row_from:row_to,col_from:col_to]
    return section.reshape(-1,1,row_to-row_from, col_to-col_from)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


The second cell contains the training loop. For every iteration (`j`) it does the following:

1. **Training pass.** For each batch of 128 training images, the images are reshaped to 28 x 28, all 3 x 3 sections are collected with `get_image_section`, and `flattened_input.dot(kernels)` gives `kernel_output`, the output of every kernel at every position.
2. **Forward propagation.** `layer_1` is the `tanh` of the kernel outputs, reshaped so that each image occupies one row. A random 0/1 mask (`dropout_mask`) is applied to it (the kept values are doubled), and `layer_2` is the `softmax` of the product with `weights_1_2`. The number of correct training predictions is counted in `correct_cnt`.
3. **Learning.** The output delta is the difference between the one-hot labels and `layer_2`, divided by `batch_size * layer_2.shape[0]`. It is propagated back through `weights_1_2` and `tanh2deriv`, and the same mask is applied again. `weights_1_2` is updated with `alpha` times the product of `layer_1.T` and the output delta. For the kernels, the hidden delta is reshaped to the shape of `kernel_output` and multiplied by `flattened_input.T`; this product adds up the contributions of **every position of every image**, which is what it means that all positions share the same kernel weights. The result `k_update` is then applied to `kernels` with `alpha`.
4. **Test pass.** After the training batches, every test image is processed one at a time in the same way (without the random mask and without softmax), and the number of correct predictions is stored in `test_correct_cnt`.
5. **Progress report.** The test and training accuracies of the iteration are written to the screen.

# **I deliberately stopped running the cell below because it took up a lot of time**

In [2]:
for j in range(iterations):
    correct_cnt = 0
    for i in range(int(len(images) / batch_size)):
        batch_start, batch_end=((i * batch_size),((i+1)*batch_size))
        layer_0 = images[batch_start:batch_end]
        layer_0 = layer_0.reshape(layer_0.shape[0],28,28)
        layer_0.shape

        sects = list()
        for row_start in range(layer_0.shape[1]-kernel_rows):
            for col_start in range(layer_0.shape[2] - kernel_cols):
                sect = get_image_section(layer_0,
                                         row_start,
                                         row_start+kernel_rows,
                                         col_start,
                                         col_start+kernel_cols)
                sects.append(sect)

        expanded_input = np.concatenate(sects,axis=1)
        es = expanded_input.shape
        flattened_input = expanded_input.reshape(es[0]*es[1],-1)

        kernel_output = flattened_input.dot(kernels)
        layer_1 = tanh(kernel_output.reshape(es[0],-1))
        dropout_mask = np.random.randint(2,size=layer_1.shape)
        layer_1 *= dropout_mask * 2
        layer_2 = softmax(np.dot(layer_1,weights_1_2))

        for k in range(batch_size):
            labelset = labels[batch_start+k:batch_start+k+1]
            _inc = int(np.argmax(layer_2[k:k+1]) ==
                                np.argmax(labelset))
            correct_cnt += _inc

        layer_2_delta = (labels[batch_start:batch_end]-layer_2)\
                        / (batch_size * layer_2.shape[0])
        layer_1_delta = layer_2_delta.dot(weights_1_2.T) * \
                        tanh2deriv(layer_1)
        layer_1_delta *= dropout_mask
        weights_1_2 += alpha * layer_1.T.dot(layer_2_delta)
        l1d_reshape = layer_1_delta.reshape(kernel_output.shape)
        k_update = flattened_input.T.dot(l1d_reshape)
        kernels -= alpha * k_update

    test_correct_cnt = 0

    for i in range(len(test_images)):

        layer_0 = test_images[i:i+1]
        layer_0 = layer_0.reshape(layer_0.shape[0],28,28)
        layer_0.shape

        sects = list()
        for row_start in range(layer_0.shape[1]-kernel_rows):
            for col_start in range(layer_0.shape[2] - kernel_cols):
                sect = get_image_section(layer_0,
                                         row_start,
                                         row_start+kernel_rows,
                                         col_start,
                                         col_start+kernel_cols)
                sects.append(sect)

        expanded_input = np.concatenate(sects,axis=1)
        es = expanded_input.shape
        flattened_input = expanded_input.reshape(es[0]*es[1],-1)

        kernel_output = flattened_input.dot(kernels)
        layer_1 = tanh(kernel_output.reshape(es[0],-1))
        layer_2 = np.dot(layer_1,weights_1_2)

        test_correct_cnt += int(np.argmax(layer_2) ==
                                np.argmax(test_labels[i:i+1]))
    if(j % 1 == 0):
        sys.stdout.write("\n"+ \
            "I:" + str(j) + \
            " Test-Acc:"+str(test_correct_cnt/float(len(test_images)))+\
            " Train-Acc:" + str(correct_cnt/float(len(images))))


I:0 Test-Acc:0.0414 Train-Acc:0.094
I:1 Test-Acc:0.0502 Train-Acc:0.039
I:2 Test-Acc:0.0425 Train-Acc:0.059
I:3 Test-Acc:0.0472 Train-Acc:0.051
I:4 Test-Acc:0.0546 Train-Acc:0.065
I:5 Test-Acc:0.0688 Train-Acc:0.094
I:6 Test-Acc:0.101 Train-Acc:0.108
I:7 Test-Acc:0.1563 Train-Acc:0.117
I:8 Test-Acc:0.2312 Train-Acc:0.146
I:9 Test-Acc:0.3 Train-Acc:0.169
I:10 Test-Acc:0.365 Train-Acc:0.196
I:11 Test-Acc:0.4163 Train-Acc:0.222
I:12 Test-Acc:0.454 Train-Acc:0.279
I:13 Test-Acc:0.476 Train-Acc:0.276
I:14 Test-Acc:0.4665 Train-Acc:0.307
I:15 Test-Acc:0.3686 Train-Acc:0.247
I:16 Test-Acc:0.1923 Train-Acc:0.186
I:17 Test-Acc:0.1105 Train-Acc:0.098
I:18 Test-Acc:0.07 Train-Acc:0.07
I:19 Test-Acc:0.0479 Train-Acc:0.053
I:20 Test-Acc:0.0313 Train-Acc:0.047
I:21 Test-Acc:0.0327 Train-Acc:0.046
I:22 Test-Acc:0.0589 Train-Acc:0.045
I:23 Test-Acc:0.127 Train-Acc:0.07
I:24 Test-Acc:0.2318 Train-Acc:0.099
I:25 Test-Acc:0.3266 Train-Acc:0.163

KeyboardInterrupt: 

### **3.5 Results and Remarks**

The book shows the following output of this program (its first iterations and its last iterations):

| Iteration | Test accuracy | Train accuracy |
|---|---|---|
| 0 | 0.0288 | 0.055 |
| 1 | 0.0273 | 0.037 |
| 2 | 0.028 | 0.037 |
| 3 | 0.0292 | 0.04 |
| 4 | 0.0339 | 0.046 |
| 5 | 0.0478 | 0.068 |
| 6 | 0.076 | 0.083 |
| 7 | 0.1316 | 0.096 |
| 8 | 0.2137 | 0.127 |
| ... | ... | ... |
| 297 | 0.8774 | 0.816 |
| 298 | 0.8774 | 0.804 |
| 299 | 0.8774 | 0.814 |

The accuracy starts very low and rises quickly after the first few iterations. At the end, the test accuracy is about 87.7 %. The book concludes that **swapping the first layer of the Chapter 9 network for a convolutional layer reduces the error by another few percentage points**.

The output of the convolutional layer (`kernel_output`) is itself a series of **two-dimensional images**: the output of each kernel at each input position. This is why convolutional layers are usually **stacked**: each convolutional layer treats the output of the previous one as an input image. The book suggests trying this as a personal project and expects it to increase accuracy further.

Stacked convolutional layers were one of the main developments that made **very deep neural networks** possible and, by extension, popularized the term *deep learning*. The author stresses that this invention was a landmark for the field and that, without it, we might still be in the previous AI winter at the time of writing.

*Remark on the number of positions.* The loops use `range(layer_0.shape[1]-kernel_rows)` and `range(layer_0.shape[2] - kernel_cols)`, which for 28 x 28 images and 3 x 3 kernels visit 25 start rows and 25 start columns (625 positions per image). This agrees with the formula for `hidden_size` in the listing, (28 - 3) * (28 - 3) * 16 = 10,000. By contrast, the 8 x 8 illustration in Section 2.3 shows 6 x 6 outputs, which corresponds to visiting every possible position (8 - 3 + 1 = 6). The code is kept exactly as printed in the book.

---
## **4. Summary**

**Reusing weights is one of the most important innovations in deep learning.**

Convolutional neural networks are a more general development than one might realize. The idea behind them, reusing weights to increase accuracy, is hugely important and has an intuitive basis. To detect a cat in an image, a network must first understand colors, then lines and edges, then corners and small shapes, and finally the combinations of such low-level features that correspond to a cat. The network presumably has to learn these low-level features (such as lines and edges) as well, and the "intelligence" for detecting them is stored in the weights.

If **different weights** analyze **different parts** of an image, each group of weights must learn on its own what a line is. When one set of weights, looking at one part of the image, has learned what a line is, there is no reason to expect another set of weights, which sits in another part of the network, to be able to use that knowledge. Convolutions exploit a property of learning: when the same idea or piece of intelligence is needed in several places, the same weights should be used in those places.

> **The structure trick.** When a neural network needs to use the same idea in multiple places, try to use the same weights in all of those places. The weights become more intelligent because they receive more samples to learn from, which increases generalization.

The book adds that many of the biggest developments in deep learning of the past five years (some earlier) are variations of this idea: **convolutions, recurrent neural networks (RNNs), word embeddings and the recently published capsule networks** can all be seen through this lens. The author expects further discoveries of this kind, because it is difficult to find new, higher-level abstract ideas that a network can use repeatedly throughout its architecture.

**Key takeaways of Chapter 10:**

- **Overfitting** is closely tied to the ratio between the number of weights and the number of training datapoints. Besides regularization, one can fight it with **structure**: reusing the same weights wherever the same pattern has to be detected.
- A **convolutional layer** replaces one large dense layer by many tiny linear layers, the **kernels** (often fewer than 25 inputs and one output), each applied at every position of the input.
- A kernel slides over the image one pixel at a time, row by row, and the predictions of one kernel form a smaller square matrix. A 3 x 3 kernel on an 8 x 8 image gives a 6 x 6 matrix; four kernels give four such matrices.
- The matrices of several kernels can be combined by **sum pooling, mean pooling or max pooling**. **Max pooling** is the most popular, and only the combined matrix is passed to the next layer.
- Each kernel learns one pattern (for example a horizontal segment or a diagonal) and looks for it everywhere. Because every kernel is evaluated at many positions, its few weights are trained on many more samples, which **reduces overfitting and improves generalization**.
- In NumPy, `get_image_section` cuts out the same subregion of every image of a batch. All subregions are collected, concatenated and flattened so that each one behaves like a small image, and a single matrix product `flattened_input.dot(kernels)` evaluates all kernels at all positions.
- Replacing the first layer of the Chapter 9 network with such a layer (16 kernels of size 3 x 3) leads to a test accuracy of about 87.7 % in the book's run. Stacking convolutional layers allows very deep networks.
- **The structure trick:** if a network needs the same idea in several places, make it use the same weights there.